# Longformer with a classification head

Fine-tunes `allenai/longformer-base-4096` to decide whether two texts were written by the same
author. The hyperparameters are tuned with Optuna on a sample of the training pairs. The final
model is trained on the training pairs, its threshold is selected on the validation pairs and
it is evaluated once on the test pairs.

A restarted run continues where it stopped: the study lives in a SQLite file, every training
writes checkpoints and a finished run is not repeated. A study is never continued with other
settings than the ones it was started with.

In [ ]:
SMOKE = False  # True - a few rows of every data set, a checkpoint after every step
N_TRIALS = 30  # finished trials of the study (complete; nothing is pruned)
SEED = 42  # seed of the tuning sample, the sampler, the new head and the training order
RUN_TAG = ""  # label in the file names of a repeated run, so two runs never share a file
MODEL_ID = "allenai/longformer-base-4096"  # Hugging Face id, or the folder of a saved model
DEVICE = "cuda"  # "cpu" only for the CPU checks - there is no fallback to the CPU

Imports

In [ ]:
import gc
import logging
import re
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import optuna
import pandas as pd
import seaborn as sns
import torch
from optuna.trial import TrialState
from sklearn.metrics import confusion_matrix
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    PreTrainedModel,
    PreTrainedTokenizerBase,
    set_seed,
)

from avcore import data, evaluation, runs, tracking, transformer_training, tuning

Configuration - the model, the search space, the sizes and every file of the run

Standard libraries

In [ ]:
import gc
import logging
import re
from functools import partial
from pathlib import Path

Data, plots and Optuna

In [ ]:
import matplotlib.pyplot as plt
import optuna
import pandas as pd
import seaborn as sns
from optuna.trial import TrialState
from sklearn.metrics import confusion_matrix

Torch and Hugging Face

In [ ]:
import torch
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    PreTrainedModel,
    PreTrainedTokenizerBase,
    set_seed,
)

The shared modules of the thesis

In [ ]:
from avcore import data, evaluation, runs, tracking, transformer_training, tuning